In [1]:
#step 1 label encoding
import numpy as np

# given data
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]

#label encoding
label_map = {
    'Wine': 0,
    'Beer': 1,
    'Whiskey': 2
}

reverse_map = {v: k for k, v in label_map.items()}

# separate features and labels
X = []
y = []

for row in data:
    X.append(row[:-1])               
    y.append(label_map[row[-1]])     

X = np.array(X, dtype=float)
y = np.array(y, dtype=int)

print("X:\n", X)
print("y:\n", y)

X:
 [[12.   1.5  1. ]
 [ 5.   2.   0. ]
 [40.   0.   1. ]
 [13.5  1.2  1. ]
 [ 4.5  1.8  0. ]
 [38.   0.1  1. ]
 [11.5  1.7  1. ]
 [ 5.5  2.3  0. ]]
y:
 [0 1 2 0 1 2 0 1]


In [2]:
#geni implemented
def gini(y):
    n = len(y)
    if n == 0:
        return 0
    
    # count occurrences of each class
    classes, counts = np.unique(y, return_counts=True)
    
    # compute probabilities
    probs = counts / n
    
    # compute gini
    return 1 - np.sum(probs ** 2)

In [17]:
def best_split(X, y):
    n_samples, n_features = X.shape

    best_gini = float('inf')
    best_feature = None
    best_threshold = None

    for feature in range(n_features):
        values = X[:, feature]

        # sort unique values
        unique_values = np.sort(np.unique(values))

        # create thresholds using midpoints
        thresholds = []
        for i in range(len(unique_values) - 1):
            t = (unique_values[i] + unique_values[i + 1]) / 2
            thresholds.append(t)

        # try all thresholds
        for t in thresholds:
            left_mask = values <= t
            right_mask = values > t

            y_left = y[left_mask]
            y_right = y[right_mask]

            # skip invalid splits
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            # compute gini
            g_left = gini(y_left)
            g_right = gini(y_right)

            # weighted gini
            n = len(y)
            weighted_gini = (len(y_left)/n)*g_left + (len(y_right)/n)*g_right

            # update best split
            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature
                best_threshold = t

    return best_feature, best_threshold

In [18]:
import numpy as np

# Node class
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  


# Tree builder
class DecisionTree:
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self.build_tree(X, y, depth=0)

    def build_tree(self, X, y, depth):
        n_samples = len(y)
        n_classes = len(np.unique(y))

        # STOP CONDITIONS
        if (depth >= self.max_depth or 
            n_classes == 1 or 
            n_samples < self.min_samples_split):
            
            leaf_value = self._majority_class(y)
            return Node(value=leaf_value)

        # GET BEST SPLIT 
        feature, threshold = best_split(X, y)

        if feature is None:
            return Node(value=self._majority_class(y))

        # SPLIT
        values = X[:, feature]
        left_mask = values <= threshold
        right_mask = values > threshold

        X_left, y_left = X[left_mask], y[left_mask]
        X_right, y_right = X[right_mask], y[right_mask]

        # RECURSION
        left_child = self.build_tree(X_left, y_left, depth + 1)
        right_child = self.build_tree(X_right, y_right, depth + 1)

        # RETURN NODE
        return Node(feature_index=feature, threshold=threshold,
                    left=left_child, right=right_child)

    def _majority_class(self, y):
        values, counts = np.unique(y, return_counts=True)
        return values[np.argmax(counts)]

    # Step 5: Predict
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        # if leaf node then return class
        if node.value is not None:
            return node.value

        # check condition
        if x[node.feature_index] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)

In [20]:
#testing and evaluatiion
tree = DecisionTree(max_depth=3)
tree.fit(X, y)
test_data = np.array([
    [6.0, 2.1, 0],
    [39.0, 0.05, 1],
    [13.0, 1.3, 1]
])

preds = tree.predict(test_data)

decoded = [reverse_map[p] for p in preds]
print(decoded)

['Beer', 'Whiskey', 'Wine']
